In [1]:
import os
import shutil
import pandas as pd

# 1. Download and Extract
if not os.path.exists('CUB_200_2011'):
    !wget -q https://s3.amazonaws.com/fast-ai-imageclas/CUB_200_2011.tgz
    !tar -xzf CUB_200_2011.tgz

# 2. Define Paths
root = 'CUB_200_2011'
images_txt = pd.read_csv(os.path.join(root, 'images.txt'), sep=' ', names=['id', 'path'])
train_test_split = pd.read_csv(os.path.join(root, 'train_test_split.txt'), sep=' ', names=['id', 'is_train'])

# 3. Create Target Folders
os.makedirs('data/train', exist_ok=True)
os.makedirs('data/test', exist_ok=True)

# 4. Move Files to 'train' or 'test' folders
print("Organizing files... this may take a minute.")
for idx, row in images_txt.iterrows():
    is_train = train_test_split.iloc[idx]['is_train']
    src_path = os.path.join(root, 'images', row['path'])

    # Determine destination
    category_folder = row['path'].split('/')[0]
    if is_train:
        dst_folder = os.path.join('data/train', category_folder)
    else:
        dst_folder = os.path.join('data/test', category_folder)

    os.makedirs(dst_folder, exist_ok=True)
    shutil.copy(src_path, dst_folder)

print("Done! Data is ready in 'data/train' and 'data/test'")

Organizing files... this may take a minute.
Done! Data is ready in 'data/train' and 'data/test'


In [2]:
import random
import numpy as np
import torch
import os

def set_seed(seed=42):
    """Sets the seed for reproducibility."""
    # 1. Set seed
    random.seed(seed)

    # 2. Set seed for NumPy (for data splitting/shuffling)
    np.random.seed(seed)

    # 3. Set seed for PyTorch (CPU)
    torch.manual_seed(seed)

    # 4. Set seed for PyTorch (GPU)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # If using multi-GPU

    # 5. Ensure deterministic behavior
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f"Random seed set to {seed}")

# --- CALL THIS FUNCTION FIRST ---
set_seed(42)

Random seed set to 42


In [3]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Define ResNet50 Requirements
# Define Transforms
# TRAIN: Add augmentation (Flip, Rotate, Color) to create "new" data
train_transform = transforms.Compose([
    # REPLACE "Resize" with "RandomResizedCrop"
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)), # Zoom in (70%-100% of image)
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transform = transforms.Compose([
    # For testing, Resize larger, then Center Crop
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load Data
train_data = datasets.ImageFolder('data/train', transform=train_transform)
test_data  = datasets.ImageFolder('data/test', transform=test_transform)

# DataLoaders
train_loader = DataLoader(train_data, batch_size=32, shuffle=True, num_workers=2)
test_loader  = DataLoader(test_data, batch_size=32, shuffle=False, num_workers=2)

# Verification
print(f"Training Images: {len(train_data)}")
print(f"Testing Images:  {len(test_data)}")
print(f"Classes: {len(train_data.classes)}")

Training Images: 5994
Testing Images:  5794
Classes: 200


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models

print("Downloading Vision Transformer (ViT-B/16)")

# 1. Load Pre-trained ViT
model_vit = models.vit_b_16(pretrained=True)

# 2. Freeze the Transformer Encoder blocks (Base layers)
for param in model_vit.parameters():
    param.requires_grad = False

# 3. Replace the Classification Head for 200 Bird Classes
num_ftrs_vit = model_vit.heads.head.in_features
model_vit.heads.head = nn.Linear(num_ftrs_vit, 200)

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_vit = model_vit.to(device)

# 4. Optimizer & Scheduler (Only training the new head)
criterion = nn.CrossEntropyLoss()
optimizer_vit = optim.Adam(model_vit.heads.head.parameters(), lr=1e-3)
scheduler_vit = torch.optim.lr_scheduler.StepLR(optimizer_vit, step_size=5, gamma=0.1)

# 5. Training Loop for ViT
num_epochs_vit = 15 # 15 epochs is usually enough for a frozen ViT head
best_acc_vit = 0.0

print("Starting ViT Phase 1 Training...")

for epoch in range(num_epochs_vit):
    model_vit.train()
    running_loss = 0.0

    # Training Phase
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer_vit.zero_grad()
        outputs = model_vit(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer_vit.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)

    # Validation Phase
    model_vit.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model_vit(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total

    # Step the Scheduler
    scheduler_vit.step()

    print(f"ViT Epoch {epoch+1}/{num_epochs_vit} | Train Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.2f}%")

    # Save Best ViT Model
    if val_acc > best_acc_vit:
        best_acc_vit = val_acc
        torch.save(model_vit.state_dict(), "best_model_vit.pth")
        print(f"--> New Best ViT Model Saved! ({val_acc:.2f}%)")

print(f"ViT Training Complete. Best Accuracy: {best_acc_vit:.2f}%")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ViT_B_16_Weights.IMAGENET1K_V1`. You can also use `weights=ViT_B_16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Starting ViT Phase 1 Training...
ViT Epoch 1/15 | Train Loss: 3.2335 | Val Acc: 55.35%
--> New Best ViT Model Saved! (55.35%)
ViT Epoch 2/15 | Train Loss: 1.5424 | Val Acc: 64.64%
--> New Best ViT Model Saved! (64.64%)
ViT Epoch 3/15 | Train Loss: 1.0834 | Val Acc: 69.47%
--> New Best ViT Model Saved! (69.47%)
ViT Epoch 4/15 | Train Loss: 0.8505 | Val Acc: 71.76%
--> New Best ViT Model Saved! (71.76%)
ViT Epoch 5/15 | Train Loss: 0.6801 | Val Acc: 72.40%
--> New Best ViT Model Saved! (72.40%)
ViT Epoch 6/15 | Train Loss: 0.5386 | Val Acc: 74.37%
--> New Best ViT Model Saved! (74.37%)
ViT Epoch 7/15 | Train Loss: 0.5181 | Val Acc: 74.87%
--> New Best ViT Model Saved! (74.87%)
ViT Epoch 8/15 | Train Loss: 0.5034 | Val Acc: 75.06%
--> New Best ViT Model Saved! (75.06%)
ViT Epoch 9/15 | Train Loss: 0.5056 | Val Acc: 74.94%
ViT Epoch 10/15 | Train Loss: 0.4856 | Val Acc: 74.97%
ViT Epoch 11/15 | Train Loss: 0.4751 | Val Acc: 75.03%
ViT Epoch 12/15 | Train Loss: 0.4706 | Val Acc: 74.97%
ViT 

In [5]:
import torch.nn as nn
import torch.optim as optim
from torchvision import models

# 1. Load Pre-trained ResNet50
print("Downloading ResNet50...")
model = models.resnet50(pretrained=True)

# 2. Freeze Layers & Unfreeze Layer 4
for param in model.parameters():
    param.requires_grad = False

for param in model.layer4.parameters():
    param.requires_grad = True

# 3. Replace Head
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, 200)

# Move to GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# 4. Optimizer & Scheduler
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# Define Scheduler: Drop LR by a factor of 10 every 7 epochs
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=7, gamma=0.1)

# 5. Training Loop
num_epochs = 30
best_acc = 0.0

print("Starting training...")

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    # Training Phase
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_train_loss = running_loss / len(train_loader)

    # Validation Phase
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total

    # Step the Scheduler (Correct Placement!)
    scheduler.step()

    print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.2f}%")

    # Save Best Model
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_model.pth")
        print(f"--> New Best Model Saved! ({val_acc:.2f}%)")

print(f"Training Complete. Best Accuracy: {best_acc:.2f}%")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Starting training...
Epoch 1/30 | Train Loss: 4.1344 | Val Acc: 41.99%
--> New Best Model Saved! (41.99%)
Epoch 2/30 | Train Loss: 2.2225 | Val Acc: 59.18%
--> New Best Model Saved! (59.18%)
Epoch 3/30 | Train Loss: 1.4033 | Val Acc: 64.84%
--> New Best Model Saved! (64.84%)
Epoch 4/30 | Train Loss: 0.9864 | Val Acc: 69.62%
--> New Best Model Saved! (69.62%)
Epoch 5/30 | Train Loss: 0.7315 | Val Acc: 71.13%
--> New Best Model Saved! (71.13%)
Epoch 6/30 | Train Loss: 0.5546 | Val Acc: 71.21%
--> New Best Model Saved! (71.21%)
Epoch 7/30 | Train Loss: 0.4245 | Val Acc: 72.42%
--> New Best Model Saved! (72.42%)
Epoch 8/30 | Train Loss: 0.2708 | Val Acc: 76.94%
--> New Best Model Saved! (76.94%)
Epoch 9/30 | Train Loss: 0.2194 | Val Acc: 76.61%
Epoch 10/30 | Train Loss: 0.2012 | Val Acc: 77.55%
--> New Best Model Saved! (77.55%)
Epoch 11/30 | Train Loss: 0.1798 | Val Acc: 77.48%
Epoch 12/30 | Train Loss: 0.1740 | Val Acc: 77.72%
--> New Best Model Saved! (77.72%)
Epoch 13/30 | Train Loss: 

In [6]:
# --- PHASE 2: FULL FINE-TUNING ---
# Run this cell ONLY after the previous training loop has finished.

print("Unfreezing all layers for fine-tuning...")

# 1. Unfreeze ALL layers in ResNet50
# This allows the model to update the "feature extraction" layers (edges, textures)
# to better understand BIRD feathers specifically, rather than generic objects.
for param in model.parameters():
    param.requires_grad = True

# 2. Re-initialize Optimizer with a much LOWER learning rate
# We use 1e-5 (10x smaller) to gently nudge the weights, preserving previous knowledge.
optimizer = optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-4)

# 3. Use a Smarter Scheduler (ReduceLROnPlateau)
# This watches validation accuracy. If it stops improving for 2 epochs, it lowers the LR.
# Removed 'verbose=True' to fix the TypeError
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='max', factor=0.1, patience=2
)

# 4. Train for another 15 Epochs
fine_tune_epochs = 15
print("Starting Phase 2 Training...")

for epoch in range(fine_tune_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()

    # Validation Phase
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total

    # IMPORTANT: Update the scheduler with the new accuracy
    # This is what tells the scheduler if we are "stuck"
    scheduler.step(val_acc)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"Phase 2 Epoch {epoch+1}/{fine_tune_epochs} | Val Acc: {val_acc:.2f}% | LR: {current_lr:.2e}")

    # Save if it beats your previous best (77.93%)
    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), "best_model_finetuned.pth")
        print(f"--> New Best Model Saved! ({val_acc:.2f}%)")

print(f"Fine-Tuning Complete. Final Best Accuracy: {best_acc:.2f}%")

Unfreezing all layers for fine-tuning...
Starting Phase 2 Training...
Phase 2 Epoch 1/15 | Val Acc: 77.51% | LR: 1.00e-05
Phase 2 Epoch 2/15 | Val Acc: 78.22% | LR: 1.00e-05
Phase 2 Epoch 3/15 | Val Acc: 78.82% | LR: 1.00e-05
--> New Best Model Saved! (78.82%)
Phase 2 Epoch 4/15 | Val Acc: 78.74% | LR: 1.00e-05
Phase 2 Epoch 5/15 | Val Acc: 78.15% | LR: 1.00e-05
Phase 2 Epoch 6/15 | Val Acc: 78.63% | LR: 1.00e-06
Phase 2 Epoch 7/15 | Val Acc: 78.29% | LR: 1.00e-06
Phase 2 Epoch 8/15 | Val Acc: 78.49% | LR: 1.00e-06
Phase 2 Epoch 9/15 | Val Acc: 78.81% | LR: 1.00e-07
Phase 2 Epoch 10/15 | Val Acc: 79.00% | LR: 1.00e-07
--> New Best Model Saved! (79.00%)
Phase 2 Epoch 11/15 | Val Acc: 78.86% | LR: 1.00e-07
Phase 2 Epoch 12/15 | Val Acc: 78.77% | LR: 1.00e-07
Phase 2 Epoch 13/15 | Val Acc: 78.82% | LR: 1.00e-08
Phase 2 Epoch 14/15 | Val Acc: 78.75% | LR: 1.00e-08
Phase 2 Epoch 15/15 | Val Acc: 78.82% | LR: 1.00e-08
Fine-Tuning Complete. Final Best Accuracy: 79.00%


In [7]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

# Load the best saved model
model.load_state_dict(torch.load("best_model_finetuned.pth"))
model.eval()

y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

# Accuracy
acc = accuracy_score(y_true, y_pred)
print(f"Test Accuracy: {acc:.4f}")

# Detailed classification report (precision, recall, f1-score)
report = classification_report(y_true, y_pred, target_names=train_data.classes)
print(report)


Test Accuracy: 0.7900
                                    precision    recall  f1-score   support

        001.Black_footed_Albatross       0.80      0.80      0.80        30
              002.Laysan_Albatross       0.71      0.83      0.77        30
               003.Sooty_Albatross       0.68      0.93      0.79        28
             004.Groove_billed_Ani       0.96      0.77      0.85        30
                005.Crested_Auklet       0.86      0.86      0.86        14
                  006.Least_Auklet       0.77      0.91      0.83        11
               007.Parakeet_Auklet       0.96      0.96      0.96        23
             008.Rhinoceros_Auklet       0.68      0.72      0.70        18
              009.Brewer_Blackbird       0.50      0.55      0.52        29
          010.Red_winged_Blackbird       0.93      0.87      0.90        30
               011.Rusty_Blackbird       0.81      0.57      0.67        30
       012.Yellow_headed_Blackbird       0.96      0.88      0.92

In [8]:
# --- MULTI-SCALE TTA (Context + Standard + Detail) ---
# Strategy: Look at the bird at 3 different zoom levels to catch everything.
# 1. Wide (See whole bird)
# 2. Standard (Normal crop)
# 3. Zoom (See details)

import torch
import torch.nn as nn
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os

print("Setting up Multi-Scale Ensemble...")

model_path = "best_model_finetuned.pth"
if not os.path.exists(model_path):
    print(f"CRITICAL: '{model_path}' not found!")
else:
    # --- 1. Define the 3 Views ---

    # View A: Standard (What you have been using)
    transform_standard = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # View B: Wide / "Squish" (See the WHOLE bird, no cropping)
    # Helps if the head/tail was cut off in the standard view.
    transform_wide = transforms.Compose([
        transforms.Resize((224, 224)), # Force whole image into 224x224
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # View C: Zoom (See details)
    # Helps if the bird is small or we need to see the eye ring/beak.
    transform_zoom = transforms.Compose([
        transforms.Resize(320), # Zoom in significantly
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    # --- 2. Create 3 Loaders (Must be Shuffle=False to match up!) ---
    # We use a smaller batch size to be safe with memory
    bs = 16
    data_std  = datasets.ImageFolder('data/test', transform=transform_standard)
    data_wide = datasets.ImageFolder('data/test', transform=transform_wide)
    data_zoom = datasets.ImageFolder('data/test', transform=transform_zoom)

    loader_std  = DataLoader(data_std,  batch_size=bs, shuffle=False, num_workers=2)
    loader_wide = DataLoader(data_wide, batch_size=bs, shuffle=False, num_workers=2)
    loader_zoom = DataLoader(data_zoom, batch_size=bs, shuffle=False, num_workers=2)

    # --- 3. Load Model ---
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = models.resnet50(pretrained=False)
    model.fc = nn.Linear(model.fc.in_features, 200)
    model.load_state_dict(torch.load(model_path))
    model = model.to(device)
    model.eval()

    # --- 4. Run Multi-Scale Inference ---
    correct = 0
    total = 0
    print("Running Multi-Scale Inference (Standard + Wide + Zoom)...")

    with torch.no_grad():
        # Zip allows us to iterate 3 loaders at once
        for (img_s, lbl), (img_w, _), (img_z, _) in zip(loader_std, loader_wide, loader_zoom):
            img_s, lbl = img_s.to(device), lbl.to(device)
            img_w = img_w.to(device)
            img_z = img_z.to(device)

            # Prediction 1: Standard
            out_s = model(img_s)
            out_s_flip = model(torch.flip(img_s, dims=[3])) # Flip

            # Prediction 2: Wide (Squish)
            out_w = model(img_w)
            out_w_flip = model(torch.flip(img_w, dims=[3])) # Flip

            # Prediction 3: Zoom
            out_z = model(img_z)
            out_z_flip = model(torch.flip(img_z, dims=[3])) # Flip

            # Average ALL 6 predictions
            # We give slightly less weight to Zoom/Wide as they are "risky" views
            # Standard: 1.0, Wide: 0.8, Zoom: 0.8
            final_output = (out_s + out_s_flip +
                           (out_w * 0.8) + (out_w_flip * 0.8) +
                           (out_z * 0.8) + (out_z_flip * 0.8)) / 5.2

            _, predicted = torch.max(final_output.data, 1)
            total += lbl.size(0)
            correct += (predicted == lbl).sum().item()

    final_acc = 100 * correct / total
    print("-" * 30)
    print(f"MULTI-SCALE ACCURACY: {final_acc:.2f}%")
    print("-" * 30)

    if final_acc > 80.0:
        print("SUCCESS! Broke 80%!")
    else:
        print(f"Final Result: {final_acc:.2f}%.")

Setting up Multi-Scale Ensemble...


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Running Multi-Scale Inference (Standard + Wide + Zoom)...
------------------------------
MULTI-SCALE ACCURACY: 80.20%
------------------------------
SUCCESS! Broke 80%!


In [9]:
# ENSEMBLE APPROACH (Soft Voting)
import torch
import torch.nn as nn
from torchvision import models
import gc

print("Setting up the ResNet50 + ViT Ensemble...")

# 1. Clear memory to make room for both models
# Safely delete the old models from Colab's global memory to prevent a crash
for var in ['model', 'model_vit', 'final_output']:
    if var in globals():
        del globals()[var]

# Force garbage collection and empty the PyTorch VRAM cache
gc.collect()
torch.cuda.empty_cache()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Load the best ResNet50 Model
resnet = models.resnet50(pretrained=False) # Architecture only
resnet.fc = nn.Linear(resnet.fc.in_features, 200)
resnet.load_state_dict(torch.load("best_model_finetuned.pth"))
resnet = resnet.to(device)
resnet.eval()
print("ResNet50 Loaded.")

# 3. Load the best ViT-B/16 Model
vit = models.vit_b_16(pretrained=False)
vit.heads.head = nn.Linear(vit.heads.head.in_features, 200)
vit.load_state_dict(torch.load("best_model_vit.pth"))
vit = vit.to(device)
vit.eval()
print("ViT-B/16 Loaded.")

# 4. Run the Ensemble Evaluation
correct = 0
total = 0

print("Running Ensemble Inference. Please wait...")

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        # Get raw scores from both models
        out_resnet = resnet(images)
        out_vit = vit(images)

        # Convert raw scores to probabilities
        prob_resnet = torch.softmax(out_resnet, dim=1)
        prob_vit = torch.softmax(out_vit, dim=1)

        # SOFT VOTING: Average the probabilities
        ensemble_prob = (prob_resnet + prob_vit) / 2.0

        # The final prediction is the class with the highest average probability
        _, predicted = torch.max(ensemble_prob, dim=1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

ensemble_acc = 100 * correct / total

print("-" * 40)
print(f"RESNET50 + VIT ENSEMBLE ACCURACY: {ensemble_acc:.2f}%")
print("-" * 40)

if ensemble_acc > 80.0:
    print("Incredible result! The models complement each other perfectly.")

Setting up the ResNet50 + ViT Ensemble...
ResNet50 Loaded.
ViT-B/16 Loaded.
Running Ensemble Inference. Please wait...
----------------------------------------
RESNET50 + VIT ENSEMBLE ACCURACY: 81.86%
----------------------------------------
Incredible result! The models complement each other perfectly.
